<a href="https://colab.research.google.com/github/LIU666-sketch/Hardware-code/blob/main/Fast%20GRPO%20Fine-Tuning%20for%20Q%26A%20/qwen_grpo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Qwen-0.5B GRPO Training on Taylor Swift QA Dataset

# **1. Setting Up the Environment**

# Install necessary dependencies

In [1]:
!pip install --no-cache-dir vllm
!pip install trl datasets
!pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 MB 256.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 256.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 257.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 330.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 176.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 302.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 267.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 316.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 232.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 215.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 194.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 364.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# **2. Importing Required Libraries**


In [2]:
import re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer
from rouge_score import rouge_scorer
from peft import LoraConfig

INFO 04-07 07:37:49 [__init__.py:239] Automatically detected platform cuda.


# **3. Defining the Prompt and Dataset Preparation**


In [4]:
# Define a system prompt to maintain response consistency
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>."""

dataset_name = "openai/gsm8k"

def get_data(dataset_name, split="train",config_name="main") -> Dataset:
    """Loads and formats the dataset into a structured prompt format."""
    data = load_dataset(dataset_name,config_name)[split]
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': x['answer']
    })
    return data.select_columns(['prompt', 'answer'])

dataset = get_data(dataset_name=dataset_name)

train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

# **4. Defining Reward Functions**

In [6]:
# ROUGE-L based reward function to evaluate response similarity to reference answers
def rouge_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    responses = [completion[0]['content'] for completion in completions]
    rewards = [scorer.score(ref_answer, response)['rougeL'].fmeasure for response, ref_answer in zip(responses, answer)]
    return rewards

# Reward function based on length similarity
def length_similarity_reward_func(prompts, completions, answer, scale_factor=0.5, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    rewards = [
        min(len(response.split()), len(ref_answer.split())) / max(len(response.split()), len(ref_answer.split())) * scale_factor
        if len(ref_answer.split()) > 0 else 0.0
        for response, ref_answer in zip(responses, answer)
    ]
    return rewards

def llm_judge_reward(prompt, generated_response, reference_answer, model, tokenizer): #llm_judge_score
    """Uses the fine-tuning Qwen-0.5B model to score responses locally."""
    eval_prompt = f"""
    Evaluate the correctness of the following response compared to the reference.

    Prompt: {prompt}
    Reference Answer: {reference_answer}
    Generated Response: {generated_response}

    Score the response between 0 and 1, where:
    - 1.0 is a perfect answer.
    - 0.0 is completely incorrect.

    Provide only a numerical score with no explanation.
    """

    # Tokenize input using the already-loaded tokenizer
    inputs = tokenizer(eval_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # Ensure tensors are on the correct device

    # Generate response using the already-loaded model
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    # Decode and extract the score
    score_text = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    try:
        score = float(score_text)
        return max(0.0, min(1.0, score))  # Ensure the score is within the [0,1] range
    except:
        return 0.5  # Default fallback score if parsing fails

def llm_judge_reward(prompts, generated_responses, answer, model, tokenizer):
  """Uses Qwen-0.5B locally to evaluate response correctness."""
  eval_prompts = [ f"""Evaluate the correctness of the following response compared to the reference.

    Prompt: {p}
    Reference Answer: {r}
    Generated Response: {g}

    Score the response between 0 and 1. Only return a number."""
    for p, g, r in zip(prompts, generated_responses, answer)
]

  inputs = tokenizer(eval_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
  inputs = {k: v.to(model.device) for k, v in inputs.items()}

  with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

  scores = [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]

  try:
      return [max(0.0, min(1.0, float(s))) for s in scores]
  except:
      return [0.5] * len(prompts)  # Default fallback

def llm_judge_reward_batch(prompts, generated_responses, answer, model, tokenizer):
    """Uses the fine-tuned Qwen-0.5B model to score multiple responses at once."""

    eval_prompts = [
        f"""Evaluate the correctness of the following response compared to the reference.

        Prompt: {p}
        Reference Answer: {r}
        Generated Response: {g}

        Score the response between 0 and 1. Only return a number."""
        for p, g, r in zip(prompts, generated_responses, answer)
    ]

    # Tokenize all prompts in a batch
    inputs = tokenizer(eval_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate responses in a batch
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

    # Decode scores
    scores = [tokenizer.decode(o, skip_special_tokens=True).strip() for o in outputs]

    # Convert scores to float values
    try:
        return [max(0.0, min(1.0, float(s))) for s in scores]
    except:
        return [0.5] * len(prompts)  # Default fallback


def combined_reward(prompts, completions, answer, model, tokenizer):
    """Combines ROUGE, Length Similarity, and Qwen-0.5B as LLM-J."""

    rouge_scores = rouge_reward_func(prompts, completions, answer)
    length_scores = length_similarity_reward_func(prompts, completions, answer)

    generated_responses = [c[0]['content'] for c in completions]
    llm_scores = llm_judge_reward_batch(prompts, generated_responses, answer, model, tokenizer)

    # Weighted combination of scores
    final_rewards = [
        (0.3 * rouge) + (0.2 * length) + (0.5 * llm)
        for rouge, length, llm in zip(rouge_scores, length_scores, llm_scores)
    ]

    return final_rewards

# **5. Model and Tokenizer Initialization**


In [7]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
output_dir = "outputs/Qwen-0.5B-GRPO"

training_args = GRPOConfig(
    output_dir=output_dir,
    learning_rate=0.0001,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=100,
    lr_scheduler_type='cosine_with_restarts',
    logging_steps=1,
    bf16=True,
    per_device_train_batch_size=8, #4,
    gradient_accumulation_steps=1,
    num_generations=8, #4,
    max_prompt_length=192,
    max_completion_length=160,
    num_train_epochs=3,
    save_steps=100,
    log_on_each_node=False,
    use_vllm=True,
    vllm_gpu_memory_utilization=0.6,
    vllm_device="cuda:0",
    report_to="none"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=None
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
!trl vllm-serve --model Qwen/Qwen2.5-0.5B-Instruct --dtype half --host 0.0.0.0 --port 8000

2025-04-07 07:56:55.980239: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744012616.001778    6955 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744012616.008413    6955 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO 04-07 07:57:01 [__init__.py:239] Automatically detected platform cuda.
WARNING 04-07 07:57:05 [config.py:2704] Casting torch.bfloat16 to torch.float16.
INFO 04-07 07:57:17 [config.py:600] This model supports multiple tasks: {'reward', 'score', 'generate', 'embed', 'classify'}. Defaulting to 'generate'.
WARNING 04-07 07:57:17 [arg_utils.py:1708] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO

# **6. Fine-Tuning with GRPO**

In [ ]:

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    lora_dropout=0.1,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[lambda prompts, completions, answer: combined_reward(prompts, completions, answer, model, tokenizer)],  # Pass model & tokenizer
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)

from transformers import TrainerCallback

class AdjustContextLengthCallback(TrainerCallback):
    """Dynamically increases max_completion_length during training."""

    def on_step_begin(self, args, state, control, **kwargs):
        """Adjusts max_completion_length based on training progress."""
        step = state.global_step

        if step >= 1000:
            args.max_prompt_length = 384  # Allow longer completions
        elif step >= 500:
            args.max_completion_length = 256  # Gradually increase

        # Log changes
        if step in [500, 1000]:
            print(f"Adjusted max_completion_length to {args.max_completion_length} at step {step}")


# Add dynamic context adjustment
trainer.add_callback(AdjustContextLengthCallback())

trainer.train()

# **7. Evaluating the Model on the Test Set**

In [ ]:
import time

eval_dataset = get_data(dataset_name=dataset_name, split="test")
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
separator = ";Assistant:"

total_rouge_l = 0
total_inference_time = 0
for example in eval_dataset:
    start_time = time.time()  # Start timer
    prompt_text = "".join([d['content'] for d in example["prompt"]]) + " " + separator
    generated_text = trainer.model.generate(
        **trainer.processing_class(prompt_text, return_tensors='pt', padding=True).to('cuda')
    )
    generated_text = trainer.processing_class.batch_decode(generated_text, skip_special_tokens=True)[0]

    if separator in generated_text:
        generated_text = generated_text.split(separator, 1)[-1].strip()
    inference_time = time.time() - start_time  # Measure inference time
    total_inference_time += inference_time

    # Calculate ROUGE-L F1 score
    rouge_scores = scorer.score(example["answer"], generated_text)
    rouge_l_f1 = rouge_scores['rougeL'].fmeasure
    total_rouge_l += rouge_l_f1

average_rouge_l = total_rouge_l / len(eval_dataset)
average_inference_time = total_inference_time / len(eval_dataset)
print(f"Average ROUGE-L F1 score on test set: {average_rouge_l}")
print(f"Inference Time: {average_inference_time:.4f} sec\n")

Average ROUGE-L F1 score on test set: 0.33127232615470215
Inference Time: 1.4173 sec

